# Forecast màu sắc và dòng xe Q2/2026

Notebook này trả lời 3 câu hỏi:

1. Màu sắc nào có xu hướng tăng nhu cầu?
2. Tỷ trọng cơ cấu màu sắc dự kiến trong Q2/2026 là bao nhiêu?
3. Dòng xe nào có dấu hiệu nhu cầu giảm?

Dữ liệu sử dụng:
- `fact_sales`: dữ liệu thực tế Q1/2025 và Q1/2026.
- File forecast từ Câu 1: forecast Q2/2026 theo SKU-tháng.


In [1]:
# 1. Import và cấu hình

from pathlib import Path
import pandas as pd
import numpy as np
import psycopg2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:,.2f}".format)

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

forecast_path = project_root / "data" / "processed" / "forecasting_adjusted" / "forecast_main_q2_sku_monthly_adjusted.csv"

print("forecast_path:", forecast_path)


forecast_path: d:\tnbike-project\data\processed\forecasting_adjusted\forecast_main_q2_sku_monthly_adjusted.csv


In [2]:
# 2. Đọc forecast SKU từ Câu 1

df_fc = pd.read_csv(forecast_path)
df_fc.columns = df_fc.columns.str.strip()

text_cols = ["product_code", "product_name", "group_name", "line_name", "color"]
num_cols = [
    "forecast_qty_base",
    "forecast_qty_conservative",
    "forecast_qty_optimistic",
    "forecast_revenue_base",
    "forecast_revenue_conservative",
    "forecast_revenue_optimistic",
]

for col in text_cols:
    if col in df_fc.columns:
        df_fc[col] = df_fc[col].astype("string").str.strip().fillna(f"Unknown {col}")

for col in num_cols:
    if col in df_fc.columns:
        df_fc[col] = pd.to_numeric(df_fc[col], errors="coerce").fillna(0)

display(df_fc.head())
print("Số SKU:", df_fc["product_code"].nunique())
print("Số màu:", df_fc["color"].nunique())
print("Số dòng xe:", df_fc["product_name"].nunique())


,product_code,product_name,group_code,group_name,line_name,color,is_new_sku,missing_master_flag,missing_group_flag,missing_line_flag,missing_color_flag,qty,revenue,order_count,active_dealer_count,avg_unit_price,numeric_issue_flag,qty_outlier_flag,revenue_outlier_flag,outlier_flag,qty_lag_1,qty_lag_2,revenue_lag_1,revenue_lag_2,rolling_2m_qty,rolling_3m_qty,mom_qty_growth,mom_revenue_growth,sku_qty_share_in_group,sku_revenue_share_in_group,same_month_qty_last_year,same_month_revenue_last_year,has_sku_yoy,sku_yoy_qty_growth,sku_yoy_revenue_growth,qty_next_month,revenue_next_month,group_month_qty,group_month_revenue,fiscal_year,fiscal_month,forecast_qty_raw,forecast_revenue_raw,selected_forecast_model,forecast_selection_type,calibration_factor,forecast_qty_base,forecast_revenue_base,forecast_qty_conservative,forecast_qty_optimistic,forecast_revenue_conservative,forecast_revenue_optimistic
0,102002009000,Xe đạp Thống Nhất We Bare Bears 12 Hồng,UNKNOWN_GROUP,Unknown group,Unknown line,Hồng,1,True,True,True,False,67.00,"68,584,792.00",25.00,18.00,"1,034,272.63",0,0,0,0,28.00,3.00,"29,057,768.00","2,762,592.00",15.50,10.33,1.39,1.36,0.01,0.01,0.00,0.00,1,0.00,0.00,NaN,NaN,"6,040.00","11,686,870,390.00",2026,4,39.70,"41,060,623.40",benchmark_ewma_2m_a30,data_driven_benchmark,1.10,43.67,"45,166,685.74",34.94,52.40,"36,133,348.59","54,200,022.88"
1,102002023000,Xe đạp Thống Nhất We Bare Bears 12 Kem,UNKNOWN_GROUP,Unknown group,Unknown line,Kem,1,True,True,True,False,50.00,"44,697,024.00",21.00,12.00,"951,750.55",0,0,0,0,11.00,8.00,"11,794,070.00","9,170,368.00",9.50,6.33,3.55,2.79,0.01,0.00,0.00,0.00,1,0.00,0.00,NaN,NaN,"6,040.00","11,686,870,390.00",2026,4,22.70,"21,604,737.38",benchmark_ewma_2m_a30,data_driven_benchmark,1.10,24.97,"23,765,211.12",19.98,29.96,"19,012,168.90","28,518,253.34"
2,103002009000,Xe đạp Thống Nhất We Bare Bears 16 Hồng,UNKNOWN_GROUP,Unknown group,Unknown line,Hồng,1,True,True,True,False,78.00,"85,395,368.00",29.00,22.00,"1,182,870.33",0,0,0,0,32.00,6.00,"40,385,184.00","3,775,925.00",19.00,12.67,1.44,1.11,0.01,0.01,0.00,0.00,1,0.00,0.00,NaN,NaN,"6,040.00","11,686,870,390.00",2026,4,45.80,"54,175,461.27",benchmark_ewma_2m_a30,data_driven_benchmark,1.10,50.38,"59,593,007.39",40.30,60.46,"47,674,405.91","71,511,608.87"
3,103002023000,Xe đạp Thống Nhất We Bare Bears 16 Kem,UNKNOWN_GROUP,Unknown group,Unknown line,Kem,1,True,True,True,False,45.00,"47,647,221.00",23.00,17.00,"1,096,376.78",0,0,0,0,20.00,0.00,"25,240,740.00",0.00,10.00,6.67,1.25,0.89,0.01,0.00,0.00,0.00,1,0.00,0.00,NaN,NaN,"6,040.00","11,686,870,390.00",2026,4,27.50,"30,150,361.52",benchmark_ewma_2m_a30,data_driven_benchmark,1.10,30.25,"33,165,397.67",24.20,36.30,"26,532,318.14","39,798,477.21"
4,103002025000,Xe đạp Thống Nhất We Bare Bears 16 Xanh Mint,UNKNOWN_GROUP,Unknown group,Unknown line,Xanh Mint,1,True,True,True,False,30.00,"30,913,888.00",15.00,12.00,"1,154,506.13",0,0,0,0,9.00,1.00,"11,358,333.00","1,842,593.00",5.00,3.33,2.33,1.72,0.00,0.00,0.00,0.00,1,0.00,0.00,NaN,NaN,"6,040.00","11,686,870,390.00",2026,4,15.30,"17,663,943.84",benchmark_ewma_2m_a30,data_driven_benchmark,1.10,16.83,"19,430,338.22",13.46,20.20,"15,544,270.58","23,316,405.87"


Số SKU: 247
Số màu: 34
Số dòng xe: 247


In [3]:
# 3. Hàm tạo bảng actual + forecast theo một cấp phân tích

def quarter_from_month(month_series):
    return np.select(
        [
            month_series.isin([1, 2, 3]),
            month_series.isin([4, 5, 6]),
            month_series.isin([7, 8, 9]),
        ],
        [1, 2, 3],
        default=4
    )


def get_actual_by_level(level_col):
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="tnbike_db",
        user="postgres",
        password="123456789"
    )

    query = f"""
    SELECT
        COALESCE({level_col}, 'Unknown') AS level_value,
        fiscal_year,
        fiscal_month,
        fiscal_quarter,
        SUM(quantity) AS actual_qty
    FROM tnbike.fact_sales
    WHERE fiscal_year IN (2025, 2026)
      AND fiscal_month IN (1, 2, 3)
    GROUP BY
        COALESCE({level_col}, 'Unknown'),
        fiscal_year,
        fiscal_month,
        fiscal_quarter
    ORDER BY
        COALESCE({level_col}, 'Unknown'),
        fiscal_year,
        fiscal_month;
    """

    df = pd.read_sql(query, conn)
    conn.close()

    df["level_value"] = df["level_value"].astype("string").str.strip().fillna("Unknown")
    df["actual_qty"] = pd.to_numeric(df["actual_qty"], errors="coerce").fillna(0)
    return df


def get_forecast_by_level(df_forecast_source, level_col):
    df = df_forecast_source.copy()

    df["level_value"] = df[level_col].astype("string").str.strip().fillna("Unknown")
    df["forecast_qty_base"] = pd.to_numeric(df["forecast_qty_base"], errors="coerce").fillna(0)

    df = df[
        (df["fiscal_year"] == 2026)
        & (df["fiscal_month"].isin([4, 5, 6]))
    ].copy()

    df = (
        df
        .groupby(["level_value", "fiscal_year", "fiscal_month"], dropna=False)
        .agg(forecast_qty=("forecast_qty_base", "sum"))
        .reset_index()
    )

    df["fiscal_quarter"] = quarter_from_month(df["fiscal_month"])
    return df


def build_full_table(level_name, actual_df, forecast_df):
    actual_month = actual_df.copy()
    actual_month["month_col"] = (
        "m"
        + actual_month["fiscal_year"].astype(str)
        + "_"
        + actual_month["fiscal_month"].astype(int).astype(str).str.zfill(2)
        + "_actual"
    )

    actual_month_wide = (
        actual_month
        .pivot_table(
            index="level_value",
            columns="month_col",
            values="actual_qty",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    forecast_month = forecast_df.copy()
    forecast_month["month_col"] = (
        "m"
        + forecast_month["fiscal_year"].astype(str)
        + "_"
        + forecast_month["fiscal_month"].astype(int).astype(str).str.zfill(2)
        + "_forecast"
    )

    forecast_month_wide = (
        forecast_month
        .pivot_table(
            index="level_value",
            columns="month_col",
            values="forecast_qty",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    actual_quarter = (
        actual_df
        .groupby(["level_value", "fiscal_year", "fiscal_quarter"], dropna=False)
        .agg(actual_qty=("actual_qty", "sum"))
        .reset_index()
    )

    actual_quarter["quarter_col"] = (
        "q"
        + actual_quarter["fiscal_year"].astype(str)
        + "_q"
        + actual_quarter["fiscal_quarter"].astype(int).astype(str)
        + "_actual"
    )

    actual_quarter_wide = (
        actual_quarter
        .pivot_table(
            index="level_value",
            columns="quarter_col",
            values="actual_qty",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    forecast_quarter = (
        forecast_df
        .groupby(["level_value", "fiscal_year", "fiscal_quarter"], dropna=False)
        .agg(forecast_qty=("forecast_qty", "sum"))
        .reset_index()
    )

    forecast_quarter["quarter_col"] = (
        "q"
        + forecast_quarter["fiscal_year"].astype(str)
        + "_q"
        + forecast_quarter["fiscal_quarter"].astype(int).astype(str)
        + "_forecast"
    )

    forecast_quarter_wide = (
        forecast_quarter
        .pivot_table(
            index="level_value",
            columns="quarter_col",
            values="forecast_qty",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    base = pd.DataFrame({
        "level_value": pd.concat([
            actual_df["level_value"],
            forecast_df["level_value"]
        ])
        .dropna()
        .astype("string")
        .str.strip()
        .unique()
    })

    full_table = base.copy()
    for table in [
        actual_month_wide,
        forecast_month_wide,
        actual_quarter_wide,
        forecast_quarter_wide,
    ]:
        full_table = full_table.merge(table, on="level_value", how="left")

    final_cols = [
        "level_value",
        "m2025_01_actual",
        "m2025_02_actual",
        "m2025_03_actual",
        "m2026_01_actual",
        "m2026_02_actual",
        "m2026_03_actual",
        "m2026_04_forecast",
        "m2026_05_forecast",
        "m2026_06_forecast",
        "q2025_q1_actual",
        "q2026_q1_actual",
        "q2026_q2_forecast",
    ]

    for col in final_cols:
        if col not in full_table.columns:
            full_table[col] = 0

    full_table = full_table[final_cols].fillna(0)
    full_table = full_table.rename(columns={"level_value": level_name})

    full_table = (
        full_table
        .sort_values("q2026_q2_forecast", ascending=False)
        .reset_index(drop=True)
    )

    return full_table


## I. Màu sắc nào có xu hướng tăng nhu cầu?

Không gọi đây là “mùa vụ", vì dữ liệu chưa đủ dài để học pattern mùa vụ lặp lại. Ở đây chỉ đánh giá xu hướng nhu cầu Q2/2026 dựa trên:

- **QoQ Growth %**: Q2/2026 forecast so với Q1/2026 actual.
- **Apr-to-Jun Growth %**: xu hướng forecast từ tháng 4 đến tháng 6.

Một màu được xem là tăng nhu cầu nếu:
- `qoq_growth_pct > 0`
- `apr_to_jun_growth_pct >= 0`
- không phải màu không xác định.
Công thức:
$$
QoQ\ Growth\ \% = \frac{Q2/2026\ forecast - Q1/2026\ actual}{Q1/2026\ actual} \times 100
$$

$$
Apr\text{-}to\text{-}Jun\ Growth\ \% = \frac{Forecast\ tháng\ 6/2026 - Forecast\ tháng\ 4/2026}{Forecast\ tháng\ 4/2026} \times 100
$$



In [4]:
# 4. Bảng màu đầy đủ

color_actual = get_actual_by_level("color")
color_forecast = get_forecast_by_level(df_fc, "color")

color_full_table = build_full_table(
    level_name="color",
    actual_df=color_actual,
    forecast_df=color_forecast
)

display(color_full_table)


C:\Users\Thien Truc\AppData\Local\Temp\ipykernel_17396\3001755069.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,color,m2025_01_actual,m2025_02_actual,m2025_03_actual,m2026_01_actual,m2026_02_actual,m2026_03_actual,m2026_04_forecast,m2026_05_forecast,m2026_06_forecast,q2025_q1_actual,q2026_q1_actual,q2026_q2_forecast
0,Kem,85.00,767.00,"2,150.00","2,051.00","2,174.00","4,602.00","3,174.90","4,468.01","3,562.83","3,002.00","8,827.00","11,205.74"
1,Đen,84.00,682.00,"2,056.00","2,093.00","1,716.00","3,845.00","2,567.91","3,698.68","2,907.14","2,822.00","7,654.00","9,173.73"
2,Ghi,108.00,402.00,"1,237.00","1,589.00","1,484.00","2,733.00","1,987.31","2,650.62","2,186.30","1,747.00","5,806.00","6,824.24"
3,Hồng,147.00,423.00,"1,184.00",894.00,"1,132.00","2,082.00","1,451.80","1,968.26","1,606.74","1,754.00","4,108.00","5,026.80"
4,Trắng,127.00,334.00,"1,190.00",880.00,739.00,"2,234.00","1,305.89","2,111.11","1,547.46","1,651.00","3,853.00","4,964.45"
5,Xanh,82.00,178.00,588.00,"1,015.00",892.00,"1,423.00","1,128.77","1,404.10","1,211.37",848.00,"3,330.00","3,744.24"
6,Nâu,133.00,289.00,"1,106.00",622.00,702.00,"1,395.00",969.03,"1,342.46","1,081.06","1,528.00","2,719.00","3,392.55"
7,Xanh Mint,22.00,228.00,517.00,284.00,364.00,811.00,547.91,788.84,620.19,767.00,"1,459.00","1,956.94"
8,Xanh Dương,233.00,422.00,"1,134.00",277.00,583.00,608.00,547.31,562.35,551.82,"1,789.00","1,468.00","1,661.49"
9,Unknown color,0.00,0.00,0.00,0.00,0.00,0.00,500.72,572.95,522.39,0.00,0.00,"1,596.05"


In [5]:
# 5. Feature màu và bảng màu tăng nhu cầu

color_feature_table = color_full_table.copy()

color_feature_table["qoq_growth_pct"] = np.where(
    color_feature_table["q2026_q1_actual"] > 0,
    (
        color_feature_table["q2026_q2_forecast"]
        - color_feature_table["q2026_q1_actual"]
    )
    / color_feature_table["q2026_q1_actual"] * 100,
    np.nan
)

color_feature_table["apr_to_jun_growth_pct"] = np.where(
    color_feature_table["m2026_04_forecast"] > 0,
    (
        color_feature_table["m2026_06_forecast"]
        - color_feature_table["m2026_04_forecast"]
    )
    / color_feature_table["m2026_04_forecast"] * 100,
    np.nan
)

color_feature_table = color_feature_table[
    [
        "color",
        "qoq_growth_pct",
        "apr_to_jun_growth_pct",
    ]
].copy()

color_demand_growth = color_feature_table[
    color_feature_table["color"].notna()
    & ~color_feature_table["color"].astype(str).str.strip().isin(["Unknown", "Unknown color", "NaN", "nan", "None", ""])
    & (color_feature_table["qoq_growth_pct"] > 0)
    & (color_feature_table["apr_to_jun_growth_pct"] >= 0)
].copy()

color_demand_growth = (
    color_demand_growth
    .sort_values(["qoq_growth_pct", "apr_to_jun_growth_pct"], ascending=[False, False])
    .reset_index(drop=True)
)

display(color_demand_growth)


,color,qoq_growth_pct,apr_to_jun_growth_pct
0,Vàng Chanh,46.69,19.26
1,Tím Dạ Quang,44.60,8.91
2,Hồng Dạ Quang,40.53,18.92
3,Xanh Mint,34.13,13.19
4,Hồng Pastel,32.80,17.95
5,Xanh Da Trời,29.99,23.67
6,Trắng,28.85,18.50
7,Xanh Santorini,26.97,11.71
8,Kem,26.95,12.22
9,Coban,25.25,1.39


## II. Tỷ trọng cơ cấu màu sắc dự kiến trong Q2/2026

Tỷ trọng cơ cấu màu được tính trên **toàn bộ màu**, không chỉ các màu tăng nhu cầu.

Công thức:

$$
Share\ Q2\% = \frac{Q2/2026\ forecast\ của\ màu}{Tổng\ Q2/2026\ forecast\ của\ tất\ cả\ màu} \times 100
$$


In [6]:
# 6. Tỷ trọng cơ cấu màu sắc Q2/2026

color_mix_q2_table = color_full_table.copy()

color_mix_q2_table = color_mix_q2_table[
    color_mix_q2_table["color"].notna()
    & ~color_mix_q2_table["color"].astype(str).str.strip().isin(["Unknown", "Unknown color", "NaN", "nan", "None", ""])
].copy()

total_q2_forecast = color_mix_q2_table["q2026_q2_forecast"].sum()

color_mix_q2_table["share_q2_pct"] = np.where(
    total_q2_forecast > 0,
    color_mix_q2_table["q2026_q2_forecast"] / total_q2_forecast * 100,
    0
)

color_mix_q2_table = (
    color_mix_q2_table[
        [
            "color",
            "q2026_q2_forecast",
            "share_q2_pct",
        ]
    ]
    .sort_values("q2026_q2_forecast", ascending=False)
    .reset_index(drop=True)
)

color_mix_q2_table["share_q2_pct"] = color_mix_q2_table["share_q2_pct"].round(2).astype(str) + "%"

display(color_mix_q2_table)


,color,q2026_q2_forecast,share_q2_pct
0,Kem,"11,205.74",18.84%
1,Đen,"9,173.73",15.42%
2,Ghi,"6,824.24",11.47%
3,Hồng,"5,026.80",8.45%
4,Trắng,"4,964.45",8.35%
5,Xanh,"3,744.24",6.29%
6,Nâu,"3,392.55",5.7%
7,Xanh Mint,"1,956.94",3.29%
8,Xanh Dương,"1,661.49",2.79%
9,Xanh Pastel,"1,371.57",2.31%


## III. Dòng xe nào có dấu hiệu nhu cầu giảm hoặc nguy cơ bán chậm ?

Dữ liệu hiện tại không có tồn kho, lượng nhập hoặc vòng quay hàng tồn kho, nên không kết luận trực tiếp “bán chậm” theo nghĩa tồn kho. Phần này chỉ xác định **dòng xe có dấu hiệu nhu cầu giảm**.

Chỉ số sử dụng:

$$
Q1 \to Q2\ Growth\ \% = \frac{Q2/2026\ forecast - Q1/2026\ actual}{Q1/2026\ actual} \times 100
$$

Nếu `q1_to_q2_growth_pct < 0`, dòng xe có dấu hiệu nhu cầu giảm.


In [7]:
# 7. Bảng dòng xe đầy đủ

product_actual = get_actual_by_level("product_name")
product_forecast = get_forecast_by_level(df_fc, "product_name")

product_full_table = build_full_table(
    level_name="product_name",
    actual_df=product_actual,
    forecast_df=product_forecast
)

display(product_full_table)


C:\Users\Thien Truc\AppData\Local\Temp\ipykernel_17396\3001755069.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,product_name,m2025_01_actual,m2025_02_actual,m2025_03_actual,m2026_01_actual,m2026_02_actual,m2026_03_actual,m2026_04_forecast,m2026_05_forecast,m2026_06_forecast,q2025_q1_actual,q2026_q1_actual,q2026_q2_forecast
0,Xe đạp Thống Nhất New 26 Kem,51.00,314.00,938.00,525.00,655.00,"1,280.00",926.75,"1,263.62","1,027.81","1,303.00","2,460.00","3,218.19"
1,Xe đạp Thống Nhất New 24 Kem,9.00,204.00,583.00,371.00,546.00,825.00,692.67,843.05,737.78,796.00,"1,742.00","2,273.51"
2,Xe đạp Thống Nhất LD 26 Kem,0.00,0.00,0.00,384.00,386.00,878.00,586.96,852.15,666.52,0.00,"1,648.00","2,105.62"
3,Xe đạp Thống Nhất Puppy 20 Hồng,7.00,112.00,347.00,378.00,318.00,784.00,503.58,754.75,578.93,466.00,"1,480.00","1,837.27"
4,Xe đạp Thống Nhất LD 24-01_2023 Kem,1.00,150.00,373.00,401.00,286.00,775.00,475.97,739.54,555.04,524.00,"1,462.00","1,770.55"
5,Xe đạp Thống Nhất New 26 Café/nâu,67.00,114.00,482.00,298.00,291.00,714.00,459.69,687.69,528.09,663.00,"1,303.00","1,675.47"
6,Xe đạp Thống Nhất GN 06-26 2.0 Đen,27.00,133.00,336.00,380.00,314.00,673.00,463.87,657.37,521.92,496.00,"1,367.00","1,643.16"
7,Xe đạp Thống Nhất New 26 Trắng,60.00,145.00,468.00,253.00,212.00,680.00,387.64,639.89,463.32,673.00,"1,145.00","1,490.85"
8,Xe đạp Thống Nhất LD 26 Pastel Xanh,0.00,0.00,0.00,285.00,265.00,559.00,388.52,546.99,436.06,0.00,"1,109.00","1,371.57"
9,Xe đạp Thống Nhất GN 06-26 2.0 Ghi,6.00,122.00,269.00,309.00,204.00,572.00,345.84,544.19,405.35,397.00,"1,085.00","1,295.38"


In [8]:
# 8. Dòng xe có dấu hiệu nhu cầu giảm

product_decline_table = product_full_table.copy()

product_decline_table["q1_to_q2_growth_pct"] = np.where(
    product_decline_table["q2026_q1_actual"] > 0,
    (
        product_decline_table["q2026_q2_forecast"]
        - product_decline_table["q2026_q1_actual"]
    )
    / product_decline_table["q2026_q1_actual"] * 100,
    np.nan
)

product_decline_table["demand_decline_flag"] = np.where(
    product_decline_table["q1_to_q2_growth_pct"] < 0,
    "Demand decline",
    "No decline"
)

declining_products = (
    product_decline_table[
        product_decline_table["q1_to_q2_growth_pct"] < 0
    ][
        [
            "product_name",
            "q2026_q1_actual",
            "q2026_q2_forecast",
            "q1_to_q2_growth_pct",
            "demand_decline_flag",
        ]
    ]
    .sort_values("q1_to_q2_growth_pct", ascending=True)
    .reset_index(drop=True)
)

display(declining_products)


,product_name,q2026_q1_actual,q2026_q2_forecast,q1_to_q2_growth_pct,demand_decline_flag
0,Xe đạp Thống Nhất New 26 màu đỏ DA HP,37.00,0.00,-100.00,Demand decline
1,Xe đạp Thống Nhất New 26 Xanh DA HP,38.00,0.00,-100.00,Demand decline
2,Xe đạp Thống Nhất Neo 20-02 Coban,23.00,0.00,-100.00,Demand decline
3,Xe đạp Thống Nhất nữ 0209 Base vàng cánh gián,1.00,0.00,-100.00,Demand decline
4,Xe đạp Thống Nhất New 26 Café/nâu DA HP,37.00,0.00,-100.00,Demand decline
5,Xe đạp Thống Nhất New 26 Trắng DA HP,38.00,0.00,-100.00,Demand decline
6,Xe đạp Thống Nhất Neo 20-02 Đỏ tươi,18.00,0.00,-100.00,Demand decline
7,Xe đạp Thống Nhất Super 26 S ghi DA HP,52.00,0.00,-100.00,Demand decline
8,Xe đạp Thống Nhất Super 26 S đen DA HP,52.00,0.00,-100.00,Demand decline
9,Xe đạp Thống Nhất Super 26 S xanh DA HP,52.00,0.00,-100.00,Demand decline
